# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale

## RAG (Retrieval Augmented Generation) based on a dataset of 800,000 scraped Amazon products

#### For our 2nd agent, we will be asking OpenAI to estimate the price of one of our deals - and we will give it a hand.

We discovered that LLMs are really good at this, out of the box.

And we discovered that we can beat a frontier LLM by fine-tuning an open-source LLM.

Now we are going to try **inference time** techniques instead of training -- by using RAG!

In [1]:
import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item


In [2]:
load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(token=hf_token,add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
LITE_MODE = False

In [4]:
DB = "products_vectorstore"

In [5]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [6]:
client = chromadb.PersistentClient(path=DB)

# Introducing the SentenceTransformer Encoding LLM

The all-MiniLM is a very useful model from HuggingFace that maps sentences & paragraphs to 384 dimensional vectors and is ideal for tasks like semantic search.

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

It can run pretty quickly locally.

As an alternative, OpenAI provides a closed-source Embeddings model. Benefits compared to OpenAI embeddings:
1. It's free and fast!
3. We can run it locally, so the data never leaves our box - might be useful if you're building a personal RAG

In [7]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')

In [8]:
vector = encoder.encode("Hello, world!")
print(vector.shape)
print(vector)

(384,)
[-3.81771475e-02  3.29110920e-02 -5.45938825e-03  1.43699646e-02
 -4.02910225e-02 -1.16532452e-01  3.16876695e-02  1.91175076e-03
 -4.26223241e-02  2.91681141e-02  4.24266867e-02  3.20417136e-02
  2.98447218e-02  1.09803099e-02 -5.39396219e-02 -5.02772778e-02
 -2.35078260e-02  1.07793529e-02 -1.37707949e-01  4.11503017e-03
  2.93331016e-02  6.68411329e-02 -1.53893949e-02  4.84376438e-02
 -8.81496966e-02 -1.27268210e-02  4.14090343e-02  4.08315063e-02
 -5.01558967e-02 -5.81250601e-02  4.88015153e-02  6.88901097e-02
  5.87469004e-02  8.73099826e-03 -1.59182381e-02  8.51419494e-02
 -7.81473815e-02 -7.75168017e-02  2.07237974e-02  1.61942374e-02
  3.25106084e-02 -5.34888655e-02 -6.22287616e-02 -2.43146550e-02
  7.41278334e-03  2.39777546e-02  6.36092760e-03  5.11450805e-02
  7.27667063e-02  3.46496888e-02 -5.47711104e-02 -5.93284816e-02
 -7.16693513e-03  2.01377161e-02  3.58463563e-02  5.59092499e-03
  1.07735740e-02 -5.27637415e-02  1.01473825e-02 -8.73164646e-03
 -6.28155470e-02  

## With that background, let's populate our Chroma database

### By calculating vectors for 800,000 scraped products

This takes 30 minutes on my machine on my GPU - it might take longer for you - feel free to use the Lite dataset!

In [11]:
collection_name = "products"
existing_collection = [collection.name for collection in client.list_collections()]
if collection_name in existing_collection:
    client.delete_collection(collection_name)

# hnsw:M=8 halves graph RAM vs default 16 — essential for 16GB machines with 800k vectors
collection = client.create_collection(
    collection_name,
    metadata={"hnsw:M": 8, "hnsw:construction_ef": 100}
)

BATCH = 500  # smaller batches reduce peak memory per flush
for i in tqdm(range(0, len(train), BATCH)):
    batch = train[i:i+BATCH]
    documents = [doc.summary for doc in batch]
    vectors = encoder.encode(documents, batch_size=64, show_progress_bar=False).astype(float).tolist()
    metadatas = [{"category": item.category, "price": item.price} for item in batch]
    ids = [f"doc_{j}" for j in range(i, i + len(documents))]
    collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

  0%|          | 0/1600 [00:00<?, ?it/s]

In [9]:
collection_name = "products"
collection = client.get_or_create_collection(collection_name)

In [13]:
MAXIMUM_DATAPOINTS = 10_000

In [14]:
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [15]:
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [16]:
# prework
result = collection.get(include=["documents", "embeddings", "metadatas"], limit=MAXIMUM_DATAPOINTS)

vectors = np.array(result["embeddings"])
documents = result["documents"]
categories = [item["category"] for item in result["metadatas"]]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [17]:
#tsne
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [19]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [20]:
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [21]:
test[0]

<Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal = $219.0>

In [14]:
def vector(item):
    return encoder.encode(item.summary)

In [15]:
def find_similar(item):
    vect = vector(item)
    results = collection.query(
        query_embeddings=vect.astype(float).tolist(),
        n_results=5
    )
    documents = results["documents"][0][:]
    prices = [m['price'] for m in results["metadatas"][0][:]]
    return documents, prices


In [16]:
print(test[0])

title='Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal' category='Musical_Instruments' price=219.0 full=None weight=2.0 summary='Title: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.' prompt=None id=None


In [17]:
print(find_similar(test[0]))

(['Title: Old Blood Noise Endeavors Procession Reverb  \nCategory: Audio Effects  \nBrand: Old Blood Noise Endeavors  \nDescription: A compact, sci‑fi inspired reverb pedal with three modulation modes for creating otherworldly echo effects.  \nDetails: Features adjustable mix, decay, speed, depth, and footswitches for bypass and hold; powered by 9\u202fV DC with 60\u202fmA draw.', 'Title: Boss MD‑2 Mega Distortion Modulation Multi‑Effects Pedal  \nCategory: Music Equipment → Effects Pedals  \nBrand: Boss  \nDescription: A powerful distortion pedal that delivers extreme low‑end crunch and endless sustain for metal and hard rock.  \nDetails: Features a Gain Boost circuit, bottom‑heavy Bottom control for 6/7‑string guitars, adjustable Tone knob, and 1\u202fMΩ input impedance.', 'Title: Old Blood Noise Endeavors Mondegreen Delay Pedal  \nCategory: Musical Instruments / Effects Pedals  \nBrand: Old Blood Noise  \nDescription: A digital delay pedal that transforms your signal into creative, 

In [18]:
def make_context(similars,prices):
    message = "for context here are some items that might be similar to the item you are looking for to estimate the price.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related project {similar}\nPrice is $: {price}\n\n"
    return message


In [29]:
documents, prices = find_similar(test[0])
print(make_context(documents, prices))

for context here are some items that might be similar to the item you are looking for to estimate the price.

Potentially related project Title: Old Blood Noise Endeavors Procession Reverb  
Category: Audio Effects  
Brand: Old Blood Noise Endeavors  
Description: A compact, sci‑fi inspired reverb pedal with three modulation modes for creating otherworldly echo effects.  
Details: Features adjustable mix, decay, speed, depth, and footswitches for bypass and hold; powered by 9 V DC with 60 mA draw.
Price is $: 209.0

Potentially related project Title: Boss MD‑2 Mega Distortion Modulation Multi‑Effects Pedal  
Category: Music Equipment → Effects Pedals  
Brand: Boss  
Description: A powerful distortion pedal that delivers extreme low‑end crunch and endless sustain for metal and hard rock.  
Details: Features a Gain Boost circuit, bottom‑heavy Bottom control for 6/7‑string guitars, adjustable Tone knob, and 1 MΩ input impedance.
Price is $: 109.99

Potentially related project Title: Old B

In [19]:
def messages_for(summary,similars,prices):
    context = make_context(similars,prices)
    message = f"Estimate the price of the product, Respond with only the price and nothing else.\n\nProduct Summary: {summary}\n\n"
    message += context
    return [{"role": "user", "content": message}]

In [42]:
print(messages_for(test[0].summary,documents,prices)[0]["content"])

Estimate the price of the product, Respond with only the price and nothing else.

Product Summary: Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.

for context here are some items that might be similar to the item you are looking for to estimate the price.

Potentially related project Title: Old Blood Noise Endeavors Procession Reverb  
Category: Audio Effects  
Brand: Old Blood Noise Endeavors  
Description: A compact, sci‑fi inspired reverb pedal with three modulation modes for creating otherworldly echo effects.  
Details: Features adjustable mix, decay, speed, depth, and footswitches 

In [20]:
def gpt_5_1(item):
    documents, prices = find_similar(item)
    response = completion(model="gpt-5.1",messages=messages_for(item.summary,documents,prices))
    return response.choices[0].message.content

In [44]:
print(test[0].price)

219.0


In [45]:
print(gpt_5_1(test[0]))

229.0


In [46]:
evaluate(gpt_5_1,test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $24 $20 $20 $30 $140 $21 $1 $9 $43 $43 $99 $0 $6 $9 $3 $11 $31 $34 $30 $49 $1 $17 $15 $22 $193 $95 $2 $111 $52 $2 $5 $60 $45 $5 $70 $55 $31 $54 $12 $70 $15 $0 $5 $20 $15 $17 $1 $84 $15 $11 $10 $160 $3 $32 $32 $18 $32 $86 $3 $117 $38 $2 $55 $299 $10 $70 $226 $0 $19 $13 $13 $0 $0 $17 $13 $15 $3 $1 $6 $0 $6 $0 $35 $9 $5 $44 $36 $50 $7 $6 $1 $3 $5 $0 $31 $4 $55 $60 $105 $3 $13 $2 $10 $14 $2 $5 $265 $5 $129 $10 $11 $2 $40 $4 $10 $7 $5 $64 $87 $1 $41 $3 $4 $0 $69 $6 $31 $10 $19 $0 $33 $7 $1 $35 $0 $85 $20 $38 $12 $22 $49 $0 $10 $35 $8 $5 $60 $5 $5 $4 $22 $4 $30 $4 $6 $41 $4 $10 $2 $91 $12 $2 $1 $41 $1 $151 $15 $5 $2 $25 $3 $70 $2 $28 $21 $1 $30 $56 $8 $68 $15 $150 $31 $20 $17 $38 $14 $20 $2 $13 $6 $6 $26 $0 $5 $21 $8 $1 $1 

In [21]:
import modal
Pricer = modal.Cls.from_name("pricer-service-two","Pricer")
pricer = Pricer()

In [29]:
def specialist_agent(item):
    return pricer.price.remote(item.summary)


In [23]:
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

## Download the Neural Network weights from Week 6 into this directory

The file `deep_neural_network.pth` here:

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

In [25]:
from agents.deep_neural_network import DeepNeuralNetworkInference
runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")

In [26]:
def deep_neural_network_agent(item):
    return runner.inference(item.summary)

In [31]:
def ensemble_agent(item):
    price1 = get_price(gpt_5_1(item))
    price2 = specialist_agent(item)
    price3 = deep_neural_network_agent(item)
    return price1*0.8 + price2*0.1 + price3*0.1

In [32]:
ensemble_agent(test[0])

225.10666809082034

In [33]:
evaluate(ensemble_agent,test)

  0%|          | 0/200 [00:00<?, ?it/s]

$14 $34 $19 $11 $44 $134 $6 $2 $9 $37 $48 $93 $2 $5 $7 $2 $16 $29 $28 $18 $31 $17 $17 $44 $31 $192 $114 $0 $114 $52 $2 $6 $45 $5 $1 $78 $49 $30 $58 $8 $55 $15 $2 $20 $59 $6 $21 $1 $81 $16 $9 $26 $192 $6 $29 $22 $33 $34 $80 $8 $125 $36 $4 $52 $300 $11 $52 $271 $0 $27 $13 $11 $1 $5 $18 $12 $20 $3 $2 $5 $6 $6 $0 $41 $11 $1 $59 $12 $26 $5 $5 $3 $2 $5 $2 $27 $5 $47 $53 $153 $2 $5 $1 $21 $10 $17 $2 $266 $6 $109 $17 $11 $1 $40 $1 $5 $3 $5 $60 $96 $1 $35 $3 $5 $6 $60 $4 $29 $0 $19 $3 $25 $8 $1 $62 $0 $77 $14 $45 $12 $18 $32 $4 $13 $28 $13 $1 $3 $22 $6 $3 $27 $5 $9 $4 $19 $40 $3 $2 $3 $75 $14 $12 $1 $60 $2 $145 $16 $2 $3 $29 $2 $101 $16 $31 $19 $0 $27 $11 $9 $34 $12 $106 $27 $17 $16 $46 $13 $20 $3 $10 $5 $3 $27 $1 $10 $11 $9 $4 $7 

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)

INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using mps
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready


In [12]:
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
12:25:43 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
12:25:46 - LiteLLM:INFO: utils.py:1307 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using ollama/qwen2.5-coder:1.5b
INFO:root:[Specialist Agent] calling pricer service to calculate price
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $299.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $329.00
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $146.18
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $307.72


307.71818389892576

In [13]:
from agents.preprocessor import Preprocessor
preprocessor = Preprocessor()
preprocessor.preprocess("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

12:26:47 - LiteLLM:INFO: utils.py:3427 - 
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= qwen2.5-coder:1.5b; provider = ollama
12:26:49 - LiteLLM:INFO: utils.py:1307 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler


'Title: Shure MV7+ Professional Podcaster Microphone with USB-C and XLR Outputs\nCategory: Audio\nBrand: Shure\nDescription: A high-quality, versatile podcasting microphone that features a USB-C input for easy connectivity and XLR output for compatibility with professional audio gear.\nDetails: The MV7+ supports Bluetooth for hands-free connections, has a dynamic range of 120 dB, and offers up to 48dB gain in the microphone. It is designed with durability and protection from dust and moisture, making it suitable for outdoor use.'